### refine text to answering rag

In [1]:
from typing import List, TypedDict
import time
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from typing import Union
from pydantic import Field

C:\Users\arghy\AppData\Local\Temp\ipykernel_23248\1159099786.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
path = r"C:\Users\arghy\OneDrive\Desktop\3rd sem\1690629458.pdf"
docs = (PyPDFLoader(f'{path}').load())
# print(docs)
chunks = RecursiveCharacterTextSplitter(chunk_size = 900,chunk_overlap = 150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")
embeddings = OllamaEmbeddings(model='nomic-embed-text')
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [3]:
result = retriever.invoke('what is the syllabus of information technology humanities')

In [4]:
for i in range(len(result)):
    print(result[i].page_content)

Maulana Abul Kalam Azad University of Technology, West Bengal   
(Formerly West Bengal University of Technology)  
Syllabus for B. Tech in Information Technology   
(Applicable from the academic session 2018-2019)   
10   
PG   
   
 
Text books/ reference books:   
   
1. John E. Hopcroft, Rajeev Motwani and Jeffrey D. Ullman, Introduction to Automata Theory, 
Languages, and Computation, Pearson Education Asia.   
2. Harry R. Lewis and Christos H. Papadimitriou, Elements of the Theory of Computation, 
Pearson Education Asia.   
3. Dexter C. Kozen, Automata and Computability, Undergraduate Texts in Computer Science, 
Springer.   
4. Michael Sipser, Introduction to the Theory of Computation, PWS Publishing.   
5. John Martin, Introduction to Languages and The Theory of Computation, TataMcGraw Hill., 
PEARSON.   
6. Dr. R.B. Patel, Theory of Computation, Khanna Publishing House
Maulana Abul Kalam Azad University of Technology, West Bengal   
(Formerly West Bengal University of Technology

In [4]:
print(type(result[i].page_content))

<class 'str'>


In [5]:
import re
def decompose_to_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]

In [6]:
strip_list = []
count = 0
for i in range(len(result)):
    z=(decompose_to_sentences(result[i].page_content))
    for j in range(len(z)):
        strip_list.append(z[j])

In [7]:
query = 'syllabus of 3rd semister'
class KeepOrDrop(BaseModel):
    keepordrop:bool = Field(description='return True if the strip is important to answer as per query and False if the sentence is not that important for the query')
llm = ChatOllama(model='qwen2.5:7b',temperature=0)
llm_keepordrop = llm.with_structured_output(KeepOrDrop)
dropping_list = []
for i in range(len(strip_list)):
    z = llm_keepordrop.invoke(f'query:{query},sentence:{strip_list[i]}')
    if not z.keepordrop:
        dropping_list.append(i)

In [8]:
refined_text = ''
for i in range(len(strip_list)):
    if i not in dropping_list:
        refined_text = refined_text + '\n' + strip_list[i]

In [9]:
print(refined_text)


Maulana Abul Kalam Azad University of Technology, West Bengal (Formerly West Bengal University of Technology) Syllabus for B.
Tech in Information Technology (Applicable from the academic session 2018-2019) 10 PG Text books/ reference books: 1.
Hopcroft, Rajeev Motwani and Jeffrey D.
Ullman, Introduction to Automata Theory, Languages, and Computation, Pearson Education Asia.
Papadimitriou, Elements of the Theory of Computation, Pearson Education Asia.
Maulana Abul Kalam Azad University of Technology, West Bengal (Formerly West Bengal University of Technology) Syllabus for B.
2 Design an Adder/Subtractor composite unit.
Maulana Abul Kalam Azad University of Technology, West Bengal (Formerly West Bengal University of Technology) Syllabus for B.
Tech in Information Technology (Applicable from the academic session 2018-2019) 19 PG Design & Analysis Algorithm Lab Code: PC-IT492 Contact: 4P Name of the Course: Design & Analysis Algorithm Lab Course Code: PC-IT492 Semester: IV Duration:6 mont

In [10]:
result = llm.invoke(f'query:{query} and retrieved text:{refined_text}')

In [11]:
print(result.content)

Based on the retrieved text, it appears that the syllabus for the third semester of the B.Tech in Information Technology program at Maulana Abul Kalam Azad University of Technology, West Bengal (formerly West Bengal University of Technology) is not fully provided. However, the text does include details about a few courses and labs. Here's a summary of the information available:

### Course Details:
1. **Design & Analysis of Algorithms Lab (PC-IT492)**
   - **Semester:** IV
   - **Duration:** 6 months
   - **Maximum Marks:** 100
   - **Teaching Scheme:**
     - Theory: 4 hours/week
     - Tutorial: NIL
     - Practical: 4 hours/week
   - **Distribution of Marks:**
     - Continuous Internal Assessment: 40 marks
     - External Assessment: 60 marks
   - **Credit Points:** 2
   - **Course Outcomes:** PC-IT404.1, PC-IT404.2, PC-IT404.3
   - **Pre-Requisite:** Not specified in the provided text

2. **Environmental Sciences (MC-IT401)**
   - **Semester:** IV
   - **Duration:** 6 months
   - 

### retrieving till llm does not think that ans is good 

In [ ]:
from typing import List, TypedDict,Annotated
import time
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from langgraph.graph.message import add_messages, BaseMessage
from langchain_core.messages import HumanMessage, SystemMessage, RemoveMessage
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from typing import Union,Literal
from pydantic import Field
import re
retrieve_again = '''You are a strict retrieval quality evaluator.

Your job is to determine whether the retrieved documents are sufficient
and correctly matched to the user's query.

Return "yes" if another retrieval is necessary.

Return "yes" when:
- The retrieved documents do not contain enough information.
- The retrieved documents contain information from the wrong semester.
- The retrieved documents contain information from the wrong branch.
- Important subjects or requested fields are missing.
- The retrieved documents contain mixed or conflicting information.
- A more targeted search could improve accuracy.

Return "no" ONLY when:
- The retrieved information directly matches the requested semester/branch/topic.
- The important requested information is present.
- There is no obvious irrelevant or conflicting semester information.
- The retrieved documents are sufficient to answer accurately.

For queries asking for ALL subjects, verify that the retrieved
information appears complete rather than assuming that a few subjects
are sufficient.

Return only "yes" or "no".
'''
class KeepOrDrop(BaseModel):
    keepordrop:bool = Field(description='return True if the strip is important to answer as per query and False if the sentence is not that important for the query')
llm = ChatOllama(model='qwen2.5:7b',temperature=0)
llm_keepordrop = llm.with_structured_output(KeepOrDrop)

class enough_or_not(BaseModel):
    enoughornot: Literal['yes','no'] = Field(description=f'{retrieve_again}')
llm_retrieve_again = llm.with_structured_output(enough_or_not)



path = r"C:\Users\arghy\OneDrive\Desktop\3rd sem\1690629458.pdf"
docs = (PyPDFLoader(f'{path}').load())
# print(docs)
chunks = RecursiveCharacterTextSplitter(chunk_size = 900,chunk_overlap = 150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")
embeddings = OllamaEmbeddings(model='nomic-embed-text')
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})
class RagState(TypedDict):
    query:Annotated[list, add_messages]
    refined_retrieved_text:str
    answer:str
    retrieve_again:str
def retrieve_to_refine(state:RagState):
    query = state['query'][-1].content
    strip_list = []
    previous_text = state['refined_retrieved_text']
    count = 0
    result = retriever.invoke(f'{query}')
    for i in range(len(result)):
        z=(decompose_to_sentences(result[i].page_content))
        for j in range(len(z)):
            strip_list.append(z[j])
    dropping_list = []
    for i in range(len(strip_list)):
        z = llm_keepordrop.invoke(f'query:{query},sentence:{strip_list[i]}')
        if not z.keepordrop:
            dropping_list.append(i)
    refined_text = ''
    for i in range(len(strip_list)):
        if i not in dropping_list:
            refined_text = refined_text + '\n' + strip_list[i]
    total = previous_text+'\n'+refined_text
    return {'refined_retrieved_text':total}
def enough(state:RagState):
    query =  state['query'][-1].content
    text = state['refined_retrieved_text']
    yes_or_no = llm_retrieve_again.invoke(f'query:{query} and retrieved docs are:\n {text}')
    return {'retrieve_again':yes_or_no.enoughornot}

def route(state:RagState):
    if state['retrieve_again'].lower() == 'yes':
        return 'new_query'
    elif state['retrieve_again'].lower() == 'no':
        return 'generate'

def new_query(state:RagState):
    new_query = llm.invoke(f"""
You are a query rewriting agent for a PDF retrieval system.

The human has provided the following query:

{state["query"]}

Rewrite this query into a new, more precise search query that can retrieve additional relevant information from the PDF.

Requirements:
- Preserve the original intent of the human's query.
- Identify the key concepts, entities, keywords, and context.
- Add useful related terms that may appear in the PDF.
- Make the query more specific and retrieval-friendly.
- Do not change the meaning of the original query.
- Do not answer the question.
- Return only the rewritten query, with no explanation.
""")
    return {'query':HumanMessage(content=new_query.content)}
def generate(state:RagState):
    result = llm.invoke(f'generate answer as per the query:{state["query"][0]} and the retrieve documents:{state["refined_retrieved_text"]}').content_blocks
    return {'answer':result}
def decompose_to_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]
graph = StateGraph(RagState)
graph.add_node('retrieve',retrieve_to_refine)
graph.add_node('enough',enough)
graph.add_node('generate',generate)
graph.add_node('new_query',new_query)

graph.add_edge(START,'retrieve')
graph.add_edge('retrieve','enough')
graph.add_conditional_edges('enough',route)
graph.add_edge('new_query','retrieve')
graph.add_edge('generate',END)
workflow = graph.compile()
# while True:
query = input('USER:')
# strip_list = []
# count = 0
# result = retriever.invoke(f'{query}')
# for i in range(len(result)):
#     z=(decompose_to_sentences(result[i].page_content))
#     for j in range(len(z)):
#         strip_list.append(z[j])
# dropping_list = []
# for i in range(len(strip_list)):
#     z = llm_keepordrop.invoke(f'query:{query},sentence:{strip_list[i]}')
#     if not z.keepordrop:
#         dropping_list.append(i)
# refined_text = ''
# for i in range(len(strip_list)):
#     if i not in dropping_list:
#         refined_text = refined_text + '\n' + strip_list[i]


result = workflow.invoke({'query':query,'refined_retrieved_text':''})
print('AI:',result)

AI: {'query': [HumanMessage(content='give me the syllabus of 3rd semister of all subject of information technology', additional_kwargs={}, response_metadata={}, id='56ef5b62-7f0b-41c3-b440-e8a0e68145f8'), HumanMessage(content='give me the syllabus for all subjects in the 3rd semester of information technology including course codes and details', additional_kwargs={}, response_metadata={}, id='ec8ce821-5d7d-439d-9218-17b362e1a97b'), HumanMessage(content='Retrieve the syllabus for all subjects in the 3rd semester of Information Technology, including course codes, details, and any related topics such as course objectives, assessment methods, and recommended textbooks.', additional_kwargs={}, response_metadata={}, id='9078973e-d7a0-4ac4-9108-1d2ed09a3386')], 'refined_retrieved_text': '\n\nMaulana Abul Kalam Azad University of Technology, West Bengal (Formerly West Bengal University of Technology) Syllabus for B.\nTech in Information Technology (Applicable from the academic session 2018-201

In [29]:
result['query']

[HumanMessage(content='give me the syllabus of 3rd semister of all subject of information technology', additional_kwargs={}, response_metadata={}, id='56ef5b62-7f0b-41c3-b440-e8a0e68145f8'),
 HumanMessage(content='give me the syllabus for all subjects in the 3rd semester of information technology including course codes and details', additional_kwargs={}, response_metadata={}, id='ec8ce821-5d7d-439d-9218-17b362e1a97b'),
 HumanMessage(content='Retrieve the syllabus for all subjects in the 3rd semester of Information Technology, including course codes, details, and any related topics such as course objectives, assessment methods, and recommended textbooks.', additional_kwargs={}, response_metadata={}, id='9078973e-d7a0-4ac4-9108-1d2ed09a3386')]

In [31]:
result['answer'][0]['text']

'Based on the provided syllabus for the B.Tech in Information Technology at Maulana Abul Kalam Azad University of Technology, West Bengal (formerly West Bengal University of Technology), the syllabus for the 3rd semester includes the following subjects:\n\n1. **Design & Analysis of Algorithms (PC-IT404)**\n   - **Duration:** 6 months\n   - **Maximum Marks:** 100\n   - **Teaching Scheme:** 3 hours of theory per week\n   - **Examination Scheme:**\n     - Mid Semester Exam: 15 marks\n     - Tutorial: NIL\n     - Assignment and Quiz: 10 marks\n     - Attendance: 5 marks\n     - Practical: 3 hours per week\n     - End Semester Exam: 70 marks\n   - **Credit Points:** 3\n   - **Course Outcomes:**\n     - PC-IT404.1\n     - PC-IT404.2\n     - PC-IT404.3\n   - **Pre-Requisite:** Pre-Requisite as in PC-IT404\n\n2. **Analog & Digital Electronics (ES-IT301)**\n   - **Duration:** 6 months\n   - **Maximum Marks:** 100\n   - **Teaching Scheme:** 3 hours of theory per week\n   - **Examination Scheme:*

In [ ]:
'Based on the provided syllabus for the B.Tech in Information Technology at Maulana Abul Kalam Azad University of Technology, West Bengal (formerly West Bengal University of Technology), the syllabus for the 3rd semester includes the following subjects:\n\n1. **Design & Analysis of Algorithms (PC-IT404)**\n   - **Duration:** 6 months\n   - **Maximum Marks:** 100\n   - **Teaching Scheme:** 3 hours of theory per week\n   - **Examination Scheme:**\n     - Mid Semester Exam: 15 marks\n     - Tutorial: NIL\n     - Assignment and Quiz: 10 marks\n     - Attendance: 5 marks\n     - Practical: 3 hours per week\n     - End Semester Exam: 70 marks\n   - **Credit Points:** 3\n   - **Course Outcomes:**\n     - PC-IT404.1\n     - PC-IT404.2\n     - PC-IT404.3\n   - **Pre-Requisite:** Pre-Requisite as in PC-IT404\n\n2. **Analog & Digital Electronics (ES-IT301)**\n   - **Duration:** 6 months\n   - **Maximum Marks:** 100\n   - **Teaching Scheme:** 3 hours of theory per week\n   - **Examination Scheme:**\n     - Mid Semester Exam: 15 marks\n     - Tutorial: NIL\n     - Assignment and Quiz: 10 marks\n     - Attendance: 5 marks\n     - Practical: 3 hours per week\n     - End Semester Exam: 70 marks\n   - **Credit Points:** 3\n   - **Laboratory Experiments:**\n     - Familiarity with IC-chips: a) Multiplexer, b) Decoder, c) Encoder\n     - Comparator Truth Table verification and clarification from Data-book\n\n3. **Environmental Science and Engineering (JGEC)**\n   - **Duration:** 6 months\n   - **Maximum Marks:** 100\n   - **Teaching Scheme:** 3 hours of theory per week\n   - **Examination Scheme:**\n     - Mid Semester Exam: 15 marks\n     - Tutorial: NIL\n     - Assignment and Quiz: 10 marks\n     - Attendance: 5 marks\n     - Practical: 3 hours per week\n     - End Semester Exam: 70 marks\n   - **Credit Points:** 2\n   - **Course Outcomes:**\n     - PC-IT402.1\n     - PC-IT402.2\n     - PC-IT402.3\n     - PC-IT402.4\n   - **Pre-Requisite:** Pre-requisites as in PC-IT402\n\n4. **Project III (Proj PR-IT881)**\n   - **Duration:** 0 hours\n   - **Maximum Marks:** 12\n   - **Teaching Scheme:** 0 hours\n   - **Examination Scheme:**\n     - Viva: 0 marks\n   - **Credit Points:** 12\n   - **Total Credit:** 27\n\n5. **Open Elective III (OE-IT702)**\n   - **Duration:** 3 hours of theory per week\n   - **Maximum Marks:** 21\n   - **Teaching Scheme:** 3 hours of theory per week\n   - **Examination Scheme:**\n     - Mid Semester Exam: 15 marks\n     - Tutorial: NIL\n     - Assignment and Quiz: 10 marks\n     - Attendance: 5 marks\n     - Practical: 12 hours\n     - End Semester Exam: 12 marks\n   - **Credit Points:** 17\n\nThe syllabus for the 3rd semester of Information Technology at this university covers a range of topics including design and analysis of algorithms, analog and digital electronics, environmental science and engineering, and project work.'